# Banc d'essai `bench` — génération AP-HP

Notebook du nouveau monde `work_prompts/` (spec : `docs/spec_testrun_run_stage.md`, v3.4).
L'ancien monde `work_modif_prompts/` n'est pas touché.

Le geste, toujours le même :

1. **scénarios** produits par la chaîne fictomed (locale), posés une fois en
   graine du test (`seed_user_prompts`) — un dossier autonome par scénario ;
2. **montage** du jeu de templates du test (`shutil.copytree` des sources vers `tests/NN/system/<position>/`) ;
3. **édition** des `.txt` du jeu du test — jamais des sources ;
4. **figement** par scénario (`copy_system_prompts`) ;
5. **contrôle à sec** (`dry_run=True`) puis **run réel** (`generate`) — relancer autant que nécessaire.

Rien ne se nettoie : `clean_existing_prompts` n'existe plus. Pour repartir de zéro,
on crée `tests/NN+1`. Le disque fait foi ; `work_prompts/tests/` est gitignoré (§9).

La clé API vient **exclusivement** de l'environnement (`MISTRAL_API_KEY`), jamais du
notebook. Toutes les cellules jusqu'aux dry-run inclus s'exécutent **sans clé**.

## 0. Bootstrap — racine du repo dans `sys.path`, import explicite de `bench`

In [2]:
from __future__ import annotations

from pathlib import Path
import os
import shutil
import sys

import polars as pl
from IPython.display import display


def _find_repo_root(start: Path) -> Path:
    """Racine du repo Stream : le dossier qui contient `bench/` et `core/`."""
    for candidate in (start, *start.parents):
        if (candidate / "bench").is_dir() and (candidate / "core").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Racine du repo Stream introuvable depuis {start} — lancer le "
        "notebook depuis work_prompts/ (ou un sous-dossier du repo)."
    )


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import bench
from bench import (
    BenchError,
    Pricing,
    copy_system_prompts,
    generate,
    load_reports,
    scenario_dirs,
    seed_user_prompts,
    summarize_costs,
    user_from_column,
    write_prompts,
)

WORK_DIR = REPO_ROOT / "work_prompts"
TESTS_DIR = WORK_DIR / "tests"  # gitignoré (§9)
TEMPLATE_ONE_GEN = WORK_DIR / "template_one_gen"
TEMPLATE_FIRST_GEN = WORK_DIR / "template_first_gen"
TEMPLATE_SECOND_GEN = WORK_DIR / "template_second_gen"


def show_first_prompt(result, *, max_chars: int = 6000) -> None:
    """Affiche le premier prompt assemblé d'un `GenResult` dry-run.

    Contrôle à sec ET test de complétude : si `generate` a rendu la main,
    aucun fichier ne manquait dans aucun dossier scénario (§3.5).
    """
    if result.reports.height == 0:
        print("Aucun scénario retenu.")
        return
    row = result.reports.row(0, named=True)
    print(f"Scénario : {row['scenario']} — famille : {row['template']}")
    for label, key in (
        ("PROMPT SYSTÈME", "system_prompt"),
        ("PROMPT USER", "user_prompt"),
        ("PREFIX (assistant)", "prefix"),
    ):
        text = row[key] or ""
        print(f"\n{'=' * 28} {label} — {len(text)} caractère(s) {'=' * 28}")
        print(text[:max_chars])
        if len(text) > max_chars:
            print(f"[... tronqué à {max_chars} caractères]")


print("Racine repo :", REPO_ROOT)
print("bench       :", Path(bench.__file__).resolve().parent)
for _path in (TEMPLATE_ONE_GEN, TEMPLATE_FIRST_GEN, TEMPLATE_SECOND_GEN):
    print(f"{_path.name:20s}:", "OK" if _path.is_dir() else f"ABSENT ({_path})")

Racine repo : /Users/remi/Documents/Stream
bench       : /Users/remi/Documents/Stream/bench
template_one_gen    : OK
template_first_gen  : OK
template_second_gen : OK


## 1. Clé API — jamais en dur

La clé vient **exclusivement** de `os.environ["MISTRAL_API_KEY"]`, exportée **avant**
le lancement du kernel (`export MISTRAL_API_KEY=...` puis relancer Jupyter). Elle ne
doit jamais apparaître dans le notebook, ni dans aucun fichier versionné.

Le client n'est construit qu'au moment d'un **run réel** : les cellules dry-run
n'ont pas besoin de clé.

In [3]:
def mistral_client():
    """Client Mistral construit à la demande — uniquement pour les runs réels."""
    try:
        api_key = os.environ["MISTRAL_API_KEY"]
    except KeyError:
        raise RuntimeError(
            "Variable d'environnement MISTRAL_API_KEY absente.\n"
            "Exporter la clé AVANT de lancer le kernel :\n"
            "    export MISTRAL_API_KEY=...    # puis relancer jupyter\n"
            "La clé ne doit JAMAIS être écrite dans ce notebook ni dans un "
            "fichier versionné."
        ) from None
    if not api_key.strip():
        raise RuntimeError("MISTRAL_API_KEY est définie mais vide.")
    from core.clients import MistralClient

    return MistralClient(api_key=api_key)


print("MISTRAL_API_KEY présente :", "MISTRAL_API_KEY" in os.environ)

MISTRAL_API_KEY présente : True


## 2. Paramètres de génération

In [4]:
MODEL = os.environ.get("MISTRAL_MODEL", "mistral-large-latest")
MAX_TOKENS_SUMMARY = 8_000
MAX_TOKENS_CR = 128_000
PRICING = Pricing(
    batch_input_usd_per_million=0.25,
    batch_output_usd_per_million=0.75,
)

# Prefill de la première génération d'un test deux temps (repris de l'ancien
# notebook).
FIRST_GEN_PREFIX = "Résumé clinique :"

# Injection du résumé intermédiaire dans le user prompt du second temps (§4).
SUMMARY_HEADER = """### RÉSUMÉ CLINIQUE ISSU DE LA PREMIÈRE GÉNÉRATION
Le résumé ci-dessous est une aide intermédiaire.
Le scénario clinique, les codes, les fiches descriptives et les instructions
restent prioritaires en cas de divergence."""

SUMMARY_FOOTER = "### FIN DU RÉSUMÉ CLINIQUE INTERMÉDIAIRE"

# Test 3 — vérificateur : textes d'exemple, à adapter à la campagne.
VERIF_SYSTEM = """Tu es un médecin DIM. On te fournit un compte rendu
hospitalier généré automatiquement. Vérifie sa cohérence clinique et sa
conformité aux règles de codage, puis rends un verdict structuré :
CONFORME ou NON CONFORME, suivi de la liste des anomalies constatées."""

VERIF_USER = "Vérifie le compte rendu suivant et rends ton verdict."
VERIF_HEADER = "### COMPTE RENDU À VÉRIFIER"
VERIF_FOOTER = "### FIN DU COMPTE RENDU"

print("Modèle :", MODEL)
print("Tarifs batch ($ / 1M tokens) :", PRICING)

Modèle : mistral-large-latest
Tarifs batch ($ / 1M tokens) : Pricing(batch_input_usd_per_million=0.25, batch_output_usd_per_million=0.75)


## 3. Scénarios — chaîne fictomed (voie nominale)

Les parquets de `data/aphp` contiennent des **profils sources** (PMSI), pas des
scénarios : c'est la chaîne fictomed qui produit les scénarios
(`generation_id`, `template_name`, `user_prompt`, `prefix`, ...). Elle est
entièrement **locale** — Mistral n'intervient qu'aux `generate()`, aucune clé
n'est nécessaire ici. Les fonctions sont importées de
`work_modif_prompts/aphp_generation_utils.py`, réutilisées **telles quelles** ;
leurs fichiers de travail vont sous `TD/.fictomed/` (dossier caché, hors
découverte). La frontière est nette : fictomed produit les scénarios, `bench`
commence après.

> **Prérequis** : fictomed installé en **éditable** depuis le clone
> `work_modif_prompts/_dependencies/fictomed_prompt_work` (branche
> `prompt-work`), comme le faisait le §1 de l'ancien notebook :
> `pip install -e work_modif_prompts/_dependencies/fictomed_prompt_work`.
> Le paquet PyPI `fictomed` 0.1.2 n'embarque pas `regles_atih.yml` et fait
> échouer la génération ; après un `uv sync` (qui réinstalle la version
> PyPI), refaire l'installation éditable.

In [5]:
# Paramètres de la sélection — à renseigner.
SOURCE_PROFILES_PATH = REPO_ROOT / "data/aphp/scenarios_bn_all_20260128.pq"  # à renseigner

SOURCE_FILTERS: list[dict] = [
    # ex. {"column": "diag2", "op": "endswith", "value": "8"},
]
CANDIDATE_POOL_SIZE = 200

SCENARIO_FILTERS: list[dict] = [
    # ex. {"column": "template_name", "op": "eq", "value": "surgery_outpatient.txt"},
]
TARGET_N = 10          # garde-fou : on est dans du test
RANDOM_SELECTION = True
RANDOM_SEED = 42

In [6]:
import importlib.util

_spec_utils = importlib.util.spec_from_file_location(
    "aphp_generation_utils",
    REPO_ROOT / "work_modif_prompts" / "aphp_generation_utils.py",
)
aphp_utils = importlib.util.module_from_spec(_spec_utils)
_prev_dwb = sys.dont_write_bytecode
sys.dont_write_bytecode = True  # pas de __pycache__ dans l'ancien monde
try:
    _spec_utils.loader.exec_module(aphp_utils)
finally:
    sys.dont_write_bytecode = _prev_dwb
print("Utilitaires amont importés depuis :", _spec_utils.origin)

Utilitaires amont importés depuis : /Users/remi/Documents/Stream/work_modif_prompts/aphp_generation_utils.py


### `servers.yaml` et fichiers de travail

fictomed lit ses chemins (profils d'entrée, sorties, référentiels) dans un
fichier de config `servers.yaml`, **généré ici pour le run** sous
`TD/.fictomed/`. Pendant la génération, le fichier de profils actif est
temporairement remplacé par les candidats sélectionnés puis **restauré**
(bloc `finally` déjà dans `generate_and_select_fictomed_scenarios`, backup
sous `.fictomed/_backups/`).

In [7]:
TD = TESTS_DIR / "01"            # le test que ces scénarios vont servir
FICTOMED_DIR = TD / ".fictomed"  # caché => hors découverte (§2)
(FICTOMED_DIR / "_backups").mkdir(parents=True, exist_ok=True)

_, _, _, candidate_source = aphp_utils.prepare_source_candidates(
    source_profiles_path=SOURCE_PROFILES_PATH,
    source_filters=SOURCE_FILTERS,
    candidate_pool_size=CANDIDATE_POOL_SIZE,
    random_selection=RANDOM_SELECTION,
    random_seed=RANDOM_SEED,
)

aphp_utils.write_fictomed_config(
    config_file=FICTOMED_DIR / "servers.yaml",
    project_root=REPO_ROOT,
    run_dir=FICTOMED_DIR,
)

_, _, selected_scenarios = aphp_utils.generate_and_select_fictomed_scenarios(
    candidate_source=candidate_source,
    config_file=FICTOMED_DIR / "servers.yaml",
    aphp_data_dir=REPO_ROOT / "data" / "aphp",
    paths={"backups": FICTOMED_DIR / "_backups"},
    scenario_filters=SCENARIO_FILTERS,
    target_n=TARGET_N,
    random_selection=RANDOM_SELECTION,
    random_seed=RANDOM_SEED,
    run_dir=FICTOMED_DIR,
)

Fichier source : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq
Shape          : (142912, 14)
Colonnes       : ['mode_hospit', 'sexe', 'age', 'racine', 'ghm2', 'diag2', 'mdp', 'nbda', 'diagnostic_associes', 'n', 'mode_entree', 'mode_sortie', 'agean', 'duree']

Candidats envoyés à fictomed : (200, 16)
Configuration fictomed : /Users/remi/Documents/Stream/work_prompts/tests/01/.fictomed/servers.yaml

pipelines:
  brest:
    data:
      input: /Users/remi/Documents/Stream/data/brest
      output: /Users/remi/Documents/Stream/work_prompts/tests/01/.fictomed/sorties/scenarios_brest
  aphp:
    data:
      input: /Users/remi/Documents/Stream/data/aphp
      output: /Users/remi/Documents/Stream/work_prompts/tests/01/.fictomed/sorties/scenarios
      referentials: /Users/remi/Documents/Stream/data/aphp/referentials

fictomed importé depuis : /Users/remi/Documents/Stream/work_modif_prompts/_dependencies/fictomed_prompt_work/fictomed/__init__.py
Dossier lu par fictomed : /Us

Construction scénarios: 100%|██████████| 3/3 [00:03<00:00,  1.17s/étape]


Scénarios sauvegardés dans /Users/remi/Documents/Stream/work_prompts/tests/01/.fictomed/sorties/scenarios/aphp_scenarios_198_20260813_152457.parquet
Profiles original restauré : /Users/remi/Documents/Stream/data/aphp/scenarios_bn_all_20260128.pq

Scénarios candidats générés : (198, 55)
source_row_id: 198 uniques / 198
source_scenario_id: 198 uniques / 198
generation_id: 198 uniques / 198

Scénarios retenus : (10, 55)
Écrit             : /Users/remi/Documents/Stream/work_prompts/tests/01/.fictomed/scenarios_fictomed_selected.parquet


In [8]:
SCENARIO_DISPLAY_COLUMNS = [
    "generation_id",
    "template_name",
    "icd_primary_code",
    "ghm2",
    "admission_type",
]
display(selected_scenarios.select(SCENARIO_DISPLAY_COLUMNS))

generation_id,template_name,icd_primary_code,ghm2,admission_type
str,str,str,str,str
"""7247e66f-0ddb-496f-a2cd-afaa8a…","""medical_inpatient.txt""","""O600""","""14Z16Z""","""HC"""
"""bae2bdc6-f4ef-48f2-887b-fbe5b0…","""medical_outpatient.txt""","""Z431""","""06M17T""","""HP"""
"""5bc0480b-8494-4d5c-a46f-90fff3…","""surgery_inpatient.txt""","""Q374""","""03C051""","""HC"""
"""89084c42-4973-44af-9dab-9423cc…","""medical_inpatient.txt""","""P073""","""15M09A""","""HC"""
"""f5e1de2d-838b-42ce-988b-9bddcc…","""surgery_inpatient.txt""","""E6606""","""10C131""","""HC"""
"""5147f4ae-17ea-4e5c-a23c-8fcf66…","""delivery_inpatient_hospit.txt""","""O640""","""14Z13A""","""HC"""
"""6d066ad1-fccc-4fc8-919f-4e3626…","""medical_inpatient.txt""","""A090""","""06M02T""","""HC"""
"""d10f77e3-de73-4d7c-a8d9-49d076…","""medical_inpatient.txt""","""P033""","""15M05A""","""HC"""
"""79a1ab26-5d6b-46cf-9e04-f7be3a…","""medical_inpatient.txt""","""I21400""","""05K051""","""HC"""


### Graine du test

Les scénarios se posent **une fois** par test (plusieurs familles admises —
spec v3.4). Si la cellule échoue avec « contient déjà des dossiers scénario »,
le test existe : continuer avec, ou créer `tests/NN+1`.

In [9]:
print(
    "Scénarios créés :",
    seed_user_prompts(TD, selected_scenarios, seed_path=SOURCE_PROFILES_PATH),
)

Scénarios créés : ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009']


## 4. Test 1 — génération directe (`tests/01`)

Le test a été seedé à la section 3 (`TD`).

In [10]:
TD1 = TD  # seedé à la section 3
print("Scénarios :", scenario_dirs(TD1))

Scénarios : ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009']


### Montage du jeu du test

**Le jeu du test s'édite LÀ : `tests/01/system/first/` — un `.txt` par famille
clinique.** Les sources `work_prompts/template_*` restent canoniques : jamais
modifiées par un test. `copytree` refuse nativement d'écraser un jeu déjà monté.

In [11]:
shutil.copytree(TEMPLATE_ONE_GEN, TD1 / "system" / "first")
print("Jeu du test monté :", TD1 / "system" / "first")

Jeu du test monté : /Users/remi/Documents/Stream/work_prompts/tests/01/system/first


### Figement

Après édition manuelle du jeu, `copy_system_prompts` fige le prompt système de
chaque scénario (résolu via son `template.txt`). Chaque dossier devient
**autonome** : il archive exactement ce qui partira au modèle.

In [12]:
print("Scénarios servis :", copy_system_prompts(TD1, "first"))

Scénarios servis : ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009']


### Contrôle à sec

Aucun appel API, aucune écriture — pas besoin de clé. Cette cellule est aussi le
**test de complétude** : `generate` échoue (`BenchError`) au moindre fichier
manquant, aucun dossier n'est sauté en silence.

In [14]:
dry1 = generate(
    TD1,
    system="prompt_system_first.txt",
    user="user_generation.txt",
    out="crh_generation.txt",
    client=None,  # inutile en dry-run
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    dry_run=True,
)
show_first_prompt(dry1)

Scénario : 0000 — famille : medical_inpatient

============================ PROMPT SYSTÈME — 13363 caractère(s) ============================
Vous êtes un médecin clinicien expert. Votre tâche est de générer un compte rendu d'hospitalisation détaillé à partir d'un scénario clinique réalisé avec des codes de la classification internationale des maladies (CIM-10) et d'autres informations décrivant l'hospitalisation.


# Contexte : le codage CIM-10

La CIM-10 est une classification des maladies, elle peut se définir comme un ensemble organisé de rubriques dans lesquelles on range des entités morbides en fonction de certains critères établis. La CIM est utilisée pour transposer les diagnostics de maladies ou autres problèmes de santé en codes alphanumériques, ce qui facilite le stockage, la recherche et l'analyse des données. Elle est très utilisée en France, en particulier pour le codage des causes de décès et pour la déclaration de l'activité hospitalière dans le cadre du programme de méd

### Run réel

Échoue immédiatement — et proprement — si `MISTRAL_API_KEY` est absente. Les
sorties `out` sont écrasées à chaque re-run (geste normal) ; chaque run réel
ajoute son entrée au journal `usage.json`.

In [ ]:
client = mistral_client()  # échoue ici, clairement, si MISTRAL_API_KEY absente

cr1 = generate(
    TD1,
    system="prompt_system_first.txt",
    user="user_generation.txt",
    out="crh_generation.txt",
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
)
print(cr1.usage)

Batch Mistral 63433dfa-721d-447b-a401-51037f7c5707 — 10 requête(s), modèle mistral-large-latest, JSONL : /Users/remi/Documents/Stream/work_prompts/tests/01/batches/crh_generation/batch_input_20260813_152612_566089.jsonl
Statut : SUCCESS — 10/10
Statut final : SUCCESS
Usage(n_requests=10, input_tokens=52833, output_tokens=24955, total_tokens=77788, input_cost_usd=0.01320825, output_cost_usd=0.01871625, total_cost_usd=0.0319245)


In [ ]:
# contrôle à sec d'abord — vérifie le prompt assemblé de la copie
dry = generate(TD, system="prompt_system_first.txt", user="user_generation.txt",
               out="crh_generation.txt", prefix_file="prefix.txt",
               only=["0001_sla"], dry_run=True,
               client=None,                # inutile en dry-run
               model=MODEL, max_tokens=MAX_TOKENS_CR, pricing=PRICING)
print(dry.reports["system_prompt"][0][-1500:])

# puis le run réel, une seule requête
cr = generate(TD, system="prompt_system_first.txt", user="user_generation.txt",
              out="crh_generation.txt", prefix_file="prefix.txt",
              only=["0001_sla"],
              client=mistral_client(),     # la clé vient de l'environnement
              model=MODEL, max_tokens=MAX_TOKENS_CR, pricing=PRICING)
print(cr.usage)


 texte). N'y recopiez jamais des synonymes de la fiche qui
n'apparaissent pas dans votre texte.

+ un dictionnaire listant les segments textuels utilisés pour décrire les informations suivantes, reproduits strictement à l’identique (copiés-collés depuis le texte généré, sans aucune modification ni interprétation, y compris ponctuation et orthographe) :
    * Date entrée
    * Date de sortie
    * Service d'hospitalisation
    * Nom/Prénom du patient
    * Nom/Prénom du médecin
    * Âge
    * Sexe
    * État général
    * Poids
    * Statut gestationnel (s'il s'agit d'une femme)
    * Gestité
    * NFS: résultats explicites (valeurs ou qualificatifs comme "normal", "élevé",...)
    * Créatinine : résultats explicites (valeurs ou qualificatifs comme "normal", "élevé",...)
    * Bilan hepatique: résultats explicites (valeurs ou qualificatifs comme "normal", "élevé",...)
    * Traitements

Par exemple : {"Âge": ["55 ans"], "Stade tumoral": ["cT4N1M2"], "Examens pour diagnostic initial": [

## 5. Test 2 — deux générations (`tests/02`)

Résumé puis CR : deux appels `generate`, le `reports` du premier nourrissant le
`context` du second (§4). Positions relatives au test : `first` ←
`template_first_gen`, `second` ← `template_second_gen`.

In [ ]:
TD2 = TESTS_DIR / "02"
print(
    "Scénarios créés :",
    seed_user_prompts(TD2, selected_scenarios, seed_path=SOURCE_PROFILES_PATH),
)

shutil.copytree(TEMPLATE_FIRST_GEN, TD2 / "system" / "first")
shutil.copytree(TEMPLATE_SECOND_GEN, TD2 / "system" / "second")
print("Jeu du test monté :", TD2 / "system")

Éditer le jeu **dans `tests/02/system/first/` et `tests/02/system/second/`**
(les sources restent canoniques), puis figer les deux positions.

In [ ]:
print("Figé first  :", copy_system_prompts(TD2, "first"))
print("Figé second :", copy_system_prompts(TD2, "second"))

### Premier temps — contrôle à sec (complétude), puis run réel

In [ ]:
dry2a = generate(
    TD2,
    system="prompt_system_first.txt",
    user="user_generation.txt",
    out="crh_resume.txt",
    client=None,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    prefix_text=FIRST_GEN_PREFIX,
    dry_run=True,
)
show_first_prompt(dry2a)

In [ ]:
client = mistral_client()

res1 = generate(
    TD2,
    system="prompt_system_first.txt",
    user="user_generation.txt",
    out="crh_resume.txt",
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    prefix_text=FIRST_GEN_PREFIX,
)
print(res1.usage)

### Second temps — contrôle à sec, puis run réel

Le contrôle à sec a besoin d'un contexte : les vrais résumés s'ils sont sur le
disque (`load_reports`), sinon un placeholder — le but est de valider
l'assemblage (§4) et la complétude des fichiers, sans clé.

In [ ]:
try:
    ctx2 = load_reports(TD2, "crh_resume.txt")
except BenchError:
    _names = scenario_dirs(TD2)
    ctx2 = pl.DataFrame(
        {
            "scenario": _names,
            "report": ["[résumé intermédiaire — placeholder de contrôle à sec]"]
            * len(_names),
        }
    )
    print("Pas de crh_resume.txt sur disque : contexte placeholder.")

dry2b = generate(
    TD2,
    system="prompt_system_second.txt",
    user="user_generation.txt",
    out="crh_final.txt",
    client=None,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    context=ctx2,
    context_header=SUMMARY_HEADER,
    context_footer=SUMMARY_FOOTER,
    dry_run=True,
)
show_first_prompt(dry2b)

In [ ]:
client = mistral_client()

cr2 = generate(
    TD2,
    system="prompt_system_second.txt",
    user="user_generation.txt",
    out="crh_final.txt",
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",  # prefill d'origine de la graine
    context=res1.reports,      # ou load_reports(TD2, "crh_resume.txt")
    context_header=SUMMARY_HEADER,
    context_footer=SUMMARY_FOOTER,
)
print(cr2.usage)

## 6. Test 3 — génération + vérificateur (sur `tests/01`)

Un passage de plus sur le même test : les prompts partagés du vérificateur sont
posés **une fois** par `write_prompts` (refus d'écraser — si la cellule échoue
parce qu'ils existent déjà, c'est voulu), puis le verdict est nourri par les CR
générés au Test 1.

In [ ]:
print("prompt_system_verif.txt :", write_prompts(TD1, "prompt_system_verif.txt", VERIF_SYSTEM))
print("user_verification.txt   :", write_prompts(TD1, "user_verification.txt", VERIF_USER))

In [ ]:
try:
    ctx1 = load_reports(TD1, "crh_generation.txt")
except BenchError:
    _names = scenario_dirs(TD1)
    ctx1 = pl.DataFrame(
        {
            "scenario": _names,
            "report": ["[CR généré — placeholder de contrôle à sec]"] * len(_names),
        }
    )
    print("Pas de crh_generation.txt sur disque : contexte placeholder.")

dry_verif = generate(
    TD1,
    system="prompt_system_verif.txt",
    user="user_verification.txt",
    out="verdict.txt",
    client=None,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    context=ctx1,
    context_header=VERIF_HEADER,
    context_footer=VERIF_FOOTER,
    dry_run=True,
)
show_first_prompt(dry_verif)

In [ ]:
client = mistral_client()

verdicts = generate(
    TD1,
    system="prompt_system_verif.txt",
    user="user_verification.txt",
    out="verdict.txt",
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    context=load_reports(TD1, "crh_generation.txt"),
    context_header=VERIF_HEADER,
    context_footer=VERIF_FOOTER,
)
print(verdicts.usage)

## 7. Itérer sur un jeu système — v2

Éditer `tests/01/system/first/*.txt`, puis re-figer sous un **autre** nom : les
variantes coexistent dans chaque dossier scénario, chaque sortie garde la
sienne (`prompt_system_first_v2.txt` → `crh_v2.txt`).

In [ ]:
# [éditer d'abord tests/01/system/first/*.txt]
print("Figé v2 :", copy_system_prompts(TD1, "first", dest="prompt_system_first_v2.txt"))

In [ ]:
dry_v2 = generate(
    TD1,
    system="prompt_system_first_v2.txt",
    user="user_generation.txt",
    out="crh_v2.txt",
    client=None,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    dry_run=True,
)
show_first_prompt(dry_v2)

In [ ]:
client = mistral_client()

cr_v2 = generate(
    TD1,
    system="prompt_system_first_v2.txt",
    user="user_generation.txt",
    out="crh_v2.txt",
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
)
print(cr_v2.usage)

## 8. Itérer par copie d'un dossier scénario

Un dossier scénario est autonome : sa copie emporte **tous** ses prompts (et ses
sorties éventuelles). Nom libre ; le prochain `generate` sur le test l'inclut
dans la découverte.

In [ ]:
_src, _dst = TD1 / "0000", TD1 / "0000_bis"
if _dst.exists():
    print("Copie déjà présente :", _dst)
else:
    shutil.copytree(_src, _dst)
    print("Copié :", _src.name, "->", _dst.name)
print("Découverte :", scenario_dirs(TD1))

## 9. Reprise de session (`load_reports`)

Kernel redémarré, rien en mémoire : le disque fait foi. `strict=False` charge
les scénarios déjà servis ; `only=` restreint le run suivant à ceux-là (run
partiel : `partial=True`, les autres dossiers restent intacts).

In [ ]:
cr_disque = load_reports(TD1, "crh_generation.txt", strict=False)
print(f"{cr_disque.height} rapport(s) sur disque pour crh_generation.txt")

if cr_disque.height:
    dry_reprise = generate(
        TD1,
        system="prompt_system_verif.txt",
        user="user_verification.txt",
        out="verdict.txt",
        client=None,
        model=MODEL,
        max_tokens=MAX_TOKENS_SUMMARY,
        pricing=PRICING,
        context=cr_disque,
        context_header=VERIF_HEADER,
        context_footer=VERIF_FOOTER,
        only=cr_disque["scenario"].to_list(),
        dry_run=True,
    )
    show_first_prompt(dry_reprise)
else:
    print("Rien à reprendre — lancer d'abord un run réel (Test 1).")

In [ ]:
client = mistral_client()

cr_disque = load_reports(TD1, "crh_generation.txt", strict=False)
if cr_disque.height == 0:
    raise RuntimeError("Rien à reprendre : aucun crh_generation.txt sur disque.")

verdicts_reprise = generate(
    TD1,
    system="prompt_system_verif.txt",
    user="user_verification.txt",
    out="verdict.txt",
    client=client,
    model=MODEL,
    max_tokens=MAX_TOKENS_SUMMARY,
    pricing=PRICING,
    context=cr_disque,
    context_header=VERIF_HEADER,
    context_footer=VERIF_FOOTER,
    only=cr_disque["scenario"].to_list(),
)
print(verdicts_reprise.usage)

## 10. `prompt_local.py` — logique de user prompt locale au test (§3.7)

Pour tester une **construction** de user prompt différente sans toucher au
package : un `prompt_local.py` à la racine du test (c'est un fichier : la
découverte l'ignore), chargé ici et passé en `user_fn=` à `seed_user_prompts`
d'un **nouveau** test. Hiérarchie des leviers : (1) éditer les `.txt` du test ;
(2) `prompt_local.py` ; (3) monkeypatch fictomed (fragile, à noter dans
`test.json["notes"]`) ; (4) modifier le clone fictomed éditable.

In [ ]:
TD3 = TESTS_DIR / "03"
TD3.mkdir(parents=True, exist_ok=True)

_prompt_local_path = TD3 / "prompt_local.py"
if not _prompt_local_path.exists():  # ne jamais écraser une version éditée
    _prompt_local_path.write_text(
        '''"""User prompt local au test 03 (spec §3.7) — exemple.

Point de départ possible : inspect.getsource sur la fonction fictomed
correspondante, copiée puis modifiée.
"""


def build_user(row: dict) -> str:
    """User prompt fictomed + rappel explicite du DP et du GHM."""
    return (
        str(row["user_prompt"]).rstrip()
        + "\\n\\nRappel codage : DP "
        + str(row.get("icd_primary_code"))
        + " — GHM "
        + str(row.get("ghm2"))
        + "\\n"
    )
''',
        encoding="utf-8",
    )
    print("Écrit :", _prompt_local_path)

import importlib.util

_spec_local = importlib.util.spec_from_file_location("prompt_local_03", _prompt_local_path)
prompt_local = importlib.util.module_from_spec(_spec_local)
_prev_dwb = sys.dont_write_bytecode
sys.dont_write_bytecode = True  # pas de __pycache__ dans le dossier de test
try:
    _spec_local.loader.exec_module(prompt_local)
finally:
    sys.dont_write_bytecode = _prev_dwb

print(
    "Scénarios créés :",
    seed_user_prompts(
        TD3,
        selected_scenarios,
        user_fn=prompt_local.build_user,
        seed_path=SOURCE_PROFILES_PATH,
    ),
)

In [ ]:
shutil.copytree(TEMPLATE_ONE_GEN, TD3 / "system" / "first")
print("Figé :", copy_system_prompts(TD3, "first"))

dry_local = generate(
    TD3,
    system="prompt_system_first.txt",
    user="user_generation.txt",
    out="crh_generation.txt",
    client=None,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    dry_run=True,
)
show_first_prompt(dry_local)

## 11. Coûts — `usage.json`

Journal **append-only** : chaque run réel ajoute une entrée, re-runs compris —
l'argent dépensé reste tracé même quand les sorties sont écrasées.
`committed_usd` = total engagé ; `current_usd` = coût de l'état courant.

In [ ]:
if TESTS_DIR.is_dir():
    for _td in sorted(TESTS_DIR.iterdir()):
        if _td.is_dir() and not _td.name.startswith("."):
            print(f"=== {_td.name} — {_td} ===")
            display(summarize_costs(_td))
else:
    print("Aucun test encore créé sous", TESTS_DIR)